7gef-a/7gef-a.pdb used as reference

Ran TMalign on terminal with ref and 5rh9-a.pdb and it worked well - trying to call the same thing here.

In [30]:
#did most of the download needed on the atomicaenv so gonna use that here
import subprocess

TMalign = "/Users/hannah/tools/TMalign/TMalign"
ref = "../aligned_files/7gef-a/7gef-a.pdb"
test = "../aligned_files/5rh9-a/5rh9-a.pdb"

subprocess.run([TMalign, ref, test])



 *********************************************************************
 * TM-align (Version 20220412): protein structure alignment          *
 * References: Y Zhang, J Skolnick. Nucl Acids Res 33, 2302-9 (2005) *
 * Please email comments and suggestions to zhanglab@zhanggroup.org   *
 *********************************************************************

Name of Chain_1: ../aligned_files/7gef-a/7gef-a.pdb (to be superimposed onto Chain_2)
Name of Chain_2: ../aligned_files/5rh9-a/5rh9-a.pdb
Length of Chain_1: 304 residues
Length of Chain_2: 304 residues

Aligned length= 304, RMSD=   0.49, Seq_ID=n_identical/n_aligned= 1.000
TM-score= 0.99432 (if normalized by length of Chain_1, i.e., LN=304, d0=6.40)
TM-score= 0.99432 (if normalized by length of Chain_2, i.e., LN=304, d0=6.40)
(You should use TM-score normalized by length of the reference structure)

(":" denotes residue pairs of d <  5.0 Angstrom, "." denotes other aligned residues)
SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNP

CompletedProcess(args=['/Users/hannah/tools/TMalign/TMalign', '../aligned_files/7gef-a/7gef-a.pdb', '../aligned_files/5rh9-a/5rh9-a.pdb'], returncode=0)

Then find the total number of aligned structures from Fragalysis

In [ ]:
import os

aligned_files = '../aligned_files'

no_of_complex = sum(
    os.path.isdir(os.path.join(aligned_files, f)) #filter hidden files
    for f in os.listdir(aligned_files)
)

print(no_of_complex)


1914


Then try to write a function: 
1. run alignment of all aligned_files to ref
2. append a list of crystal strictures if Seq_ID not equal to 1
3. include in the list TM-score as well


In [3]:
import os
import subprocess
import re

def find_mutation(
    file_path,
    ref_pdb,
    tmalign_path
):
    results = []

    # regex patterns for TM-align output
    seqid_pattern = re.compile(r"\Seq_ID=.*?=\s*([\d\.]+)")
    tmscore_pattern = re.compile(r"TM-score.*?=\s*([\d\.]+)")

    for complex_id in os.listdir(file_path):
        complex_dir = os.path.join(file_path, complex_id)
        if not os.path.isdir(complex_dir):
            continue

        test_pdb = os.path.join(complex_dir, f"{complex_id}.pdb")
        if not os.path.isfile(test_pdb):
            continue

        # skip self-alignment
        if os.path.abspath(test_pdb) == os.path.abspath(ref_pdb):
            continue

        # run TM-align
        proc = subprocess.run(
            [tmalign_path, ref_pdb, test_pdb],
            capture_output=True,
            text=True
        )

        if proc.returncode != 0:
            continue

        stdout = proc.stdout

        # parse Seq_ID and TM-score
        seqid_match = seqid_pattern.search(stdout)
        tmscore_match = tmscore_pattern.search(stdout)

        if not seqid_match or not tmscore_match:
            continue

        seq_id = float(seqid_match.group(1))
        tm_score = float(tmscore_match.group(1))

        # store only non-identical structures
        if seq_id < 1.0:
            results.append({
                "pdb_id": complex_id,
                "Seq_ID": seq_id,
                "TM_score": tm_score
            })

    return results

In [4]:
#writing my parameters again, for easier reference later
TMalign = "/Users/hannah/tools/TMalign/TMalign"
ref = "../aligned_files/7gef-a/7gef-a.pdb"
aligned_files = '../aligned_files'

mutation_results = find_mutation(aligned_files, ref, TMalign)


In [14]:
len(mutation_results)

597

In [ ]:
mutation_rate = len(mutation_results)/no_of_complex
print(mutation_rate)
#so 31% of the database downloaded from fragalysis had a version of SARS-CoV-2 Mpro with mutations to the ref Mpro

0.31174934725848563


It seems that substitution often occurs in the Mpro, but the overall fold is unlikely to be affects.
next try the same thing but with append a list of structures to exclude base on TM-score < 0.5 (unlikely to be the same fold/protein)

In [5]:
def find_non_identical_structures(
    file_path,
    ref_pdb,
    tmalign_path, 
    tm_threshold
):
    results = []

    # regex patterns for TM-align output
    seqid_pattern = re.compile(r"\Seq_ID=.*?=\s*([\d\.]+)")
    tmscore_pattern = re.compile(r"TM-score.*?=\s*([\d\.]+)")

    for complex_id in os.listdir(file_path):
        complex_dir = os.path.join(file_path, complex_id)
        if not os.path.isdir(complex_dir):
            continue

        test_pdb = os.path.join(complex_dir, f"{complex_id}.pdb")
        if not os.path.isfile(test_pdb):
            continue

        # skip self-alignment
        if os.path.abspath(test_pdb) == os.path.abspath(ref_pdb):
            continue

        # run TM-align
        proc = subprocess.run(
            [tmalign_path, ref_pdb, test_pdb],
            capture_output=True,
            text=True
        )

        if proc.returncode != 0:
            continue

        stdout = proc.stdout

        # parse Seq_ID and TM-score
        seqid_match = seqid_pattern.search(stdout)
        tmscore_match = tmscore_pattern.search(stdout)

        if not seqid_match or not tmscore_match:
            continue

        seq_id = float(seqid_match.group(1))
        tm_score = float(tmscore_match.group(1))

        # store only non-identical structures
        if tm_score < tm_threshold:
            results.append({
                "pdb_id": complex_id,
                "Seq_ID": seq_id,
                "TM_score": tm_score
            })

    return results

In [ ]:
NonIdenticalStructure = find_non_identical_structures(aligned_files, ref, TMalign, 0.5)

In [ ]:
print(NonIdenticalStructure) #no structures have TM-score lower than 0.5

[]


In [8]:
#trying 0.9
NonIdenticalStructure_9 = find_non_identical_structures(aligned_files, ref, TMalign, 0.9)

In [9]:
print(NonIdenticalStructure_9) #all TM-scores of Mpro downloaded from fragalysis are above 0.9 - same fold and likely function
# all taken forward for PeSTo transformation


[]


In [6]:
NonIdenticalStructure_95 = find_non_identical_structures(aligned_files, ref, TMalign, 0.95)
print(NonIdenticalStructure_95)

[{'pdb_id': 'MERS-a1795b', 'Seq_ID': 0.503, 'TM_score': 0.9221}, {'pdb_id': 'MERS-a2224b', 'Seq_ID': 0.503, 'TM_score': 0.92244}, {'pdb_id': 'MERS-a1094b', 'Seq_ID': 0.5, 'TM_score': 0.92077}, {'pdb_id': 'MERS-a0851a', 'Seq_ID': 0.5, 'TM_score': 0.92229}, {'pdb_id': 'MERS-a1497b', 'Seq_ID': 0.503, 'TM_score': 0.92541}, {'pdb_id': 'MERS-a1965a', 'Seq_ID': 0.503, 'TM_score': 0.92416}, {'pdb_id': 'MERS-a1753b', 'Seq_ID': 0.503, 'TM_score': 0.92487}, {'pdb_id': 'MERS-a1560a', 'Seq_ID': 0.503, 'TM_score': 0.92701}, {'pdb_id': 'MERS-a2051b', 'Seq_ID': 0.51, 'TM_score': 0.93211}, {'pdb_id': 'MERS-a2525b', 'Seq_ID': 0.51, 'TM_score': 0.93366}, {'pdb_id': 'MERS-a1417a', 'Seq_ID': 0.507, 'TM_score': 0.92445}, {'pdb_id': 'MERS-a1438a', 'Seq_ID': 0.5, 'TM_score': 0.92131}, {'pdb_id': 'MERS-a1105a', 'Seq_ID': 0.5, 'TM_score': 0.92446}, {'pdb_id': 'MERS-a2204a', 'Seq_ID': 0.51, 'TM_score': 0.9297}, {'pdb_id': 'MERS-a2688a', 'Seq_ID': 0.502, 'TM_score': 0.93074}, {'pdb_id': 'MERS-a1143b', 'Seq_ID': 0

In [ ]:
len(NonIdenticalStructure_95) #same length as the complexes with seq_ID not equal to 1

597

In [12]:
len(NonIdenticalStructure_95)/no_of_complex

0.31174934725848563

Collect all complex_id.pdb in each folder to one path for easier downstream analysis

In [47]:
import shutil

src_root = "../aligned_files"
dst_root = "../pesto_inputs"

os.makedirs(dst_root, exist_ok=True)

count = 0

for complex_id in os.listdir(src_root):
    complex_dir = os.path.join(src_root, complex_id)
    if not os.path.isdir(complex_dir):
        continue

    pdb_path = os.path.join(complex_dir, f"{complex_id}.pdb")
    if not os.path.isfile(pdb_path):
        continue

    dst_path = os.path.join(dst_root, f"{complex_id}.pdb")
    shutil.copy2(pdb_path, dst_path)
    count += 1

print(f"Copied {count} PDB files to {dst_root}")

Copied 1914 PDB files to ../pesto_inputs
